# Fine-Tuning Large Language Model

In [1]:
import numpy as np
from indra.literature import pubmed_client

In [2]:
def get_all_abstracts(list_PMID, prepend_title = True):
    metadata = pubmed_client.get_metadata_for_all_ids(list_PMID, get_abstracts=True, prepend_title=prepend_title)
    abstracts = [metadata[i]['abstract'] for i in metadata]
    return abstracts

In [3]:
positive_PMID = [
    "27661255", # Compact and highly active next-generation libraries for CRISPR-mediated gene repression and activation
    "39626969", # Structure-optimized sgRNA selection with PlatinumCRISPr for efficient Cas9 generation of knockouts
    "39019250", # Functional genomic screening in Komagataella phaffii enabled by high-activity CRISPR-Cas9 library
    "39278589", # Optimized genome-wide CRISPR screening enables rapid engineering of growth-based phenotypes in Yarrowia lipolytica
    "36576240", # Maximizing CRISPRi efficacy and accessibility with dual-sgRNA libraries and optimal effectors
    "37464007", # Optimized minimal genome-wide human sgRNA library
    "25184501", # Rational design of highly active sgRNAs for CRISPR-Cas9-mediated gene inactivation
    "26063738", # Sequence determinants of improved CRISPR sgRNA design
    "33451939", # SeqCor: correct the effect of guide RNA sequences in clustered regularly interspaced short palindromic repeats/Cas9 screening by machine learning algorithm
]

positive_abstracts = get_all_abstracts(positive_PMID, prepend_title=True)

Retrieving metadata:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving metadata: 100%|██████████| 1/1 [00:00<00:00,  1.85it/s]


In [4]:
positive_abstracts

['Compact and highly active next-generation libraries for CRISPR-mediated gene repression and activation. We recently found that nucleosomes directly block access of CRISPR/Cas9 to DNA (Horlbeck et al., 2016). Here, we build on this observation with a comprehensive algorithm that incorporates chromatin, position, and sequence features to accurately predict highly effective single guide RNAs (sgRNAs) for targeting nuclease-dead Cas9-mediated transcriptional repression (CRISPRi) and activation (CRISPRa). We use this algorithm to design next-generation genome-scale CRISPRi and CRISPRa libraries targeting human and mouse genomes. A CRISPRi screen for essential genes in K562 cells demonstrates that the large majority of sgRNAs are highly active. We also find CRISPRi does not exhibit any detectable non-specific toxicity recently observed with CRISPR nuclease approaches. Precision-recall analysis shows that we detect over 90% of essential genes with minimal false positives using a compact 5 s

In [5]:
papers = pubmed_client.get_ids("sgRNA score")
papers_abstracts = get_all_abstracts(papers, prepend_title=True)

Retrieving metadata: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]


In [6]:
len(papers_abstracts)

25

In [7]:
from openai import OpenAI

client = OpenAI()

In [20]:
def is_similar_to_positive_abstracts(abstract, positive_abstracts):
	prompt = (
		"Given the following list of positive abstracts:\n\n" +
		"\n\n".join(positive_abstracts) +
		"\n\nDetermine if the following abstract is similar to any of the positive abstracts:\n\n" +
		abstract +
		"\n\nRespond with 'Yes' or 'No'."
	)
	response = client.chat.completions.create(
		model="gpt-4o-mini",
		messages=[{"role": "user", "content": prompt}],
		max_tokens=100,
		temperature=0.7
	)
	return response.choices[0].message.content

In [21]:
# Example usage
results = []
for paper in papers_abstracts:
    results.append(is_similar_to_positive_abstracts(paper, positive_abstracts))
    

print(results)

INFO: [2025-03-29 01:10:56] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2025-03-29 01:10:57] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2025-03-29 01:10:58] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2025-03-29 01:10:59] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2025-03-29 01:10:59] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2025-03-29 01:11:01] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2025-03-29 01:11:01] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2025-03-29 01:11:02] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2025-03-29 01:11:03] httpx - HTTP Request: POST https://api.opena

['No', 'Yes', 'No', 'No', 'No', 'No', 'No.', 'Yes', 'Yes', 'No.', 'No', 'Yes', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes', 'Yes', 'No', 'No', 'No', 'No.', 'No']


In [27]:
pos_idx = np.where(np.array(results) == "Yes")[0]

In [32]:
[i.split('.')[0] for i in np.array(papers_abstracts)[pos_idx]]

['CRISPRi-Driven Genetic Screening for Designing Novel Microbial Phenotypes',
 'sgRNA-2wPSM: Identify sgRNAs on-target activity by combining two-window-based position specific mismatch and synthetic minority oversampling technique',
 'Efficient prioritization of CRISPR screen hits by accounting for targeting efficiency of guide RNA',
 'TransCrispr: Transformer Based Hybrid Model for Predicting CRISPR/Cas9 Single Guide RNA Cleavage Efficiency',
 'Key sequence features of CRISPR RNA for dual-guide CRISPR-Cas9 ribonucleoprotein complexes assembled with wild-type or HiFi Cas9',
 'Multigene editing: current approaches and beyond',
 'Multilayered VBC score predicts sgRNAs that efficiently generate loss-of-function alleles',
 'C-RNNCrispr: Prediction of CRISPR/Cas9 sgRNA activity using convolutional and recurrent neural networks']

In [33]:
positive_abstracts

['Compact and highly active next-generation libraries for CRISPR-mediated gene repression and activation. We recently found that nucleosomes directly block access of CRISPR/Cas9 to DNA (Horlbeck et al., 2016). Here, we build on this observation with a comprehensive algorithm that incorporates chromatin, position, and sequence features to accurately predict highly effective single guide RNAs (sgRNAs) for targeting nuclease-dead Cas9-mediated transcriptional repression (CRISPRi) and activation (CRISPRa). We use this algorithm to design next-generation genome-scale CRISPRi and CRISPRa libraries targeting human and mouse genomes. A CRISPRi screen for essential genes in K562 cells demonstrates that the large majority of sgRNAs are highly active. We also find CRISPRi does not exhibit any detectable non-specific toxicity recently observed with CRISPR nuclease approaches. Precision-recall analysis shows that we detect over 90% of essential genes with minimal false positives using a compact 5 s